# **Projeto Cronos**
# CHALLENGE LOCAWEB 2026

Integrantes:

* Bruno Rosa      - RM563779
* Danilo Alves    - RM564109
* Enzo Cremaschi  - RM562058
* Vinícius Macedo - RM561911

# 01 — Camada Bronze | Projeto Cronos (Locaweb Challenge 2026)

A camada Bronze é uma cópia fiel da fonte, com a **única** transformação permitida sendo tipagem mínima de datas (texto -> datetime).

**Pré-requisito:** o arquivo `utils.py` precisa estar em `/content/drive/MyDrive/cronos_project/utils.py` antes de rodar este notebook.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import sys
import pandas as pd

sys.path.append('/content/drive/MyDrive/cronos_project')

from utils import (
    setup_logging,
    PROJECT_PATHS,
    EXPECTED_COLUMNS,
    DATETIME_COLUMNS,
    load_excel_source,
    validate_schema,
    validate_categorical_domains,
    save_parquet_with_metadata,
    profile_dataframe,
    diff_row_count,
    compute_file_hash,
)

logger = setup_logging('bronze')
PROJECT_PATHS.ensure_dirs()
logger.info('Diretórios do projeto garantidos em %s', PROJECT_PATHS.root)

2026-08-15 18:14:25 | INFO     | bronze | Diretórios do projeto garantidos em /content/drive/MyDrive/cronos_project


INFO:bronze:Diretórios do projeto garantidos em /content/drive/MyDrive/cronos_project


## 1. Carga da fonte

Leitura pura do Excel, sem qualquer transformação. O hash do arquivo é calculado e registrado — é a "impressão digital" desta execução específica da Bronze.

In [4]:
source_hash = compute_file_hash(PROJECT_PATHS.raw_source)
df_bronze = load_excel_source(logger=logger)
df_bronze.head(3)

2026-08-15 18:14:27 | INFO     | bronze | Lendo fonte: /content/drive/MyDrive/cronos_project/data_raw/LW-DATASET.xlsx (sha256=87bab6e06250)


INFO:bronze:Lendo fonte: /content/drive/MyDrive/cronos_project/data_raw/LW-DATASET.xlsx (sha256=87bab6e06250)


2026-08-15 18:15:02 | INFO     | bronze | Fonte carregada: 122543 linhas x 19 colunas.


INFO:bronze:Fonte carregada: 122543 linhas x 19 colunas.


,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,Duração,Código de fechamento,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?
0,INC8654273,3 - Média,NaN,NaN,NaN,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,14,NaN,Problem: Apache Busy Workers,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
1,INC8654270,4 - Baixa,NaN,NaN,NaN,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,209,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
2,INC8654264,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,110,NaN,Problem: Alarm Application Monitoring database...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN


## 2. Validação de schema

Confere que a fonte ainda tem exatamente as colunas do dicionário de dados combinado no documento de ideação. Se a Locaweb alterar a extração (coluna renomeada, removida, ou nova), este é o ponto em que o pipeline avisa — antes de qualquer camada seguinte consumir um dado incompleto silenciosamente.

In [5]:
validate_schema(df_bronze, EXPECTED_COLUMNS, logger=logger)

2026-08-15 18:15:02 | INFO     | bronze | Schema validado: 19 colunas esperadas, todas presentes.


INFO:bronze:Schema validado: 19 colunas esperadas, todas presentes.


## 2.1 Validação de domínio categórico

Diferente da validação de schema (que checa nome de coluna), isto verifica o **conteúdo**: se `Prioridade`, `Status`, `Aberto por`, `Entrou para KPI?` e `KPI Violado?` só têm os valores já conhecidos e documentados. Isso pega um tipo de mudança de fonte que a validação de schema não pega — por exemplo, a Locaweb passar a usar uma nova categoria de Prioridade que nenhuma regra downstream foi desenhada para tratar. Não interrompe o pipeline (é um alerta, não um erro fatal), mas fica registrado no log para investigação.

In [6]:
achados_dominio = validate_categorical_domains(df_bronze, logger=logger)
if achados_dominio:
    logger.warning('Domínios com valores novos detectados — revisar antes de seguir para a Silver: %s', achados_dominio)

2026-08-15 18:15:02 | INFO     | bronze | Domínio categórico de 'Prioridade': OK, nenhum valor novo.


INFO:bronze:Domínio categórico de 'Prioridade': OK, nenhum valor novo.


2026-08-15 18:15:02 | INFO     | bronze | Domínio categórico de 'Status': OK, nenhum valor novo.


INFO:bronze:Domínio categórico de 'Status': OK, nenhum valor novo.


2026-08-15 18:15:02 | INFO     | bronze | Domínio categórico de 'Aberto por': OK, nenhum valor novo.


INFO:bronze:Domínio categórico de 'Aberto por': OK, nenhum valor novo.


2026-08-15 18:15:02 | INFO     | bronze | Domínio categórico de 'Entrou para KPI?': OK, nenhum valor novo.


INFO:bronze:Domínio categórico de 'Entrou para KPI?': OK, nenhum valor novo.


2026-08-15 18:15:02 | INFO     | bronze | Domínio categórico de 'KPI Violado?': OK, nenhum valor novo.


INFO:bronze:Domínio categórico de 'KPI Violado?': OK, nenhum valor novo.


## 3. Tipagem mínima — a única transformação permitida na Bronze

Conversão de `Aberto`, `Resolvido` e `Encerrado` de texto/objeto para `datetime`. Nada mais muda: nenhuma linha é removida, nenhum valor é alterado, nenhuma coluna é descartada.

In [8]:
df_antes = df_bronze.copy()

for col in DATETIME_COLUMNS:
    df_bronze[col] = pd.to_datetime(df_bronze[col], errors='coerce')

diff_row_count(df_antes, df_bronze, 'tipagem_datetime', logger=logger)

2026-08-15 18:21:49 | INFO     | bronze | [tipagem_datetime] 122543 -> 122543 linhas (+0, 0.00%).


INFO:bronze:[tipagem_datetime] 122543 -> 122543 linhas (+0, 0.00%).


In [9]:
n_coercoes = {
    col: int(df_bronze[col].isna().sum() - df_antes[col].isna().sum())
    for col in DATETIME_COLUMNS
}
logger.info('Valores que viraram NaT por falha de parsing (não deveria ser > 0): %s', n_coercoes)

if any(v > 0 for v in n_coercoes.values()):
    logger.warning(
        'Existem valores de data que não puderam ser convertidos — investigar antes de seguir para a Silver. '
        'Isso é um achado a registrar, não a corrigir silenciosamente aqui.'
    )

2026-08-15 18:22:22 | INFO     | bronze | Valores que viraram NaT por falha de parsing (não deveria ser > 0): {'Aberto': 0, 'Resolvido': 0, 'Encerrado': 0}


INFO:bronze:Valores que viraram NaT por falha de parsing (não deveria ser > 0): {'Aberto': 0, 'Resolvido': 0, 'Encerrado': 0}


## 4. Profiling de auditoria — "raio-x" desta execução da Bronze

Mesmo resumo estrutural usado na EDA, gerado aqui como registro formal de auditoria: o estado exato dos dados no momento em que entraram no pipeline versionado.

In [10]:
resumo_bronze = profile_dataframe(df_bronze)
resumo_bronze

,dtype,n_nao_nulos,n_nulos,pct_nulos,n_unicos
Incidente Pai,object,15127,107416,87.66,3326
Solução,object,15300,107243,87.51,2
KPI Violado?,object,25600,96943,79.11,2
Resolvido,datetime64[ns],40241,82302,67.16,37446
Código de fechamento,object,40804,81739,66.70,17
Produto,object,44608,77935,63.60,51
Categoria,object,44822,77721,63.42,141
Subcategoria,object,44823,77720,63.42,447
Item de configuração,object,120763,1780,1.45,9171
Prioridade,object,122543,0,0.00,5


In [11]:
logger.info('Shape final da Bronze: %d linhas x %d colunas.', *df_bronze.shape)
logger.info('Cobertura temporal: %s a %s.', df_bronze['Aberto'].min(), df_bronze['Aberto'].max())

2026-08-15 18:22:32 | INFO     | bronze | Shape final da Bronze: 122543 linhas x 19 colunas.


INFO:bronze:Shape final da Bronze: 122543 linhas x 19 colunas.


2026-08-15 18:22:32 | INFO     | bronze | Cobertura temporal: 2023-01-02 20:19:58 a 2025-12-31 23:45:18.


INFO:bronze:Cobertura temporal: 2023-01-02 20:19:58 a 2025-12-31 23:45:18.


## 5. Gravação

Grava em `data/bronze/lw_incidentes_raw.parquet`, com colunas de metadados de proveniência (`_ingested_at`, `_source_layer`, `_source_hash`) anexadas automaticamente pelo `utils.save_parquet_with_metadata`.


In [12]:
bronze_path = PROJECT_PATHS.bronze / 'lw_incidentes_raw.parquet'
save_parquet_with_metadata(
    df_bronze,
    bronze_path,
    source_hash=source_hash,
    layer='bronze',
    logger=logger,
)

2026-08-15 18:22:39 | INFO     | bronze | Gravado: /content/drive/MyDrive/cronos_project/data/bronze/lw_incidentes_raw.parquet (122543 linhas, 4.9 MB).


INFO:bronze:Gravado: /content/drive/MyDrive/cronos_project/data/bronze/lw_incidentes_raw.parquet (122543 linhas, 4.9 MB).


In [13]:
# Checagem de sanidade pós-gravação: reler o arquivo e confirmar shape e schema
df_checagem = pd.read_parquet(bronze_path)
assert df_checagem.shape[0] == df_bronze.shape[0], 'Divergência de linhas entre o gravado e o lido de volta!'
assert set(EXPECTED_COLUMNS).issubset(set(df_checagem.columns)), 'Colunas de negócio ausentes após a gravação!'
logger.info('Checagem pós-gravação OK: %d linhas, %d colunas (incluindo metadados).', *df_checagem.shape)

2026-08-15 18:22:40 | INFO     | bronze | Checagem pós-gravação OK: 122543 linhas, 22 colunas (incluindo metadados).


INFO:bronze:Checagem pós-gravação OK: 122543 linhas, 22 colunas (incluindo metadados).


## 6. Log de execução — resumo para o README do repositório

Célula final: gera um resumo textual curto desta execução, para colar no changelog do projeto ou no README

In [14]:
resumo_execucao = f'''
EXECUÇÃO DA CAMADA BRONZE — {pd.Timestamp.now(tz="UTC").isoformat()}
Fonte: {PROJECT_PATHS.raw_source.name} (sha256={source_hash[:12]}...)
Linhas: {df_bronze.shape[0]:,}
Colunas: {df_bronze.shape[1]} (schema validado contra o dicionário de dados)
Cobertura temporal: {df_bronze["Aberto"].min().date()} a {df_bronze["Aberto"].max().date()}
Transformação aplicada: tipagem datetime em {list(DATETIME_COLUMNS)}
Saída: {bronze_path}
'''.strip()

print(resumo_execucao)

EXECUÇÃO DA CAMADA BRONZE — 2026-08-15T18:22:42.244814+00:00
Fonte: LW-DATASET.xlsx (sha256=87bab6e06250...)
Linhas: 122,543
Colunas: 19 (schema validado contra o dicionário de dados)
Cobertura temporal: 2023-01-02 a 2025-12-31
Transformação aplicada: tipagem datetime em ['Aberto', 'Resolvido', 'Encerrado']
Saída: /content/drive/MyDrive/cronos_project/data/bronze/lw_incidentes_raw.parquet
